# Assignment

## Instructions

### Task: Building a Neural Network for Wine Classification

In this assignment, you will build and train a neural network using PyTorch to classify wine varieties based on their chemical attributes. You will use the Wine dataset, a classic machine learning dataset that contains the results of chemical analyses of wines grown in the same region in Italy but derived from three different cultivars.

#### Dataset

The Wine dataset consists of 13 features:

1. Alcohol
2. Malic acid
3. Ash
4. Alcalinity of ash
5. Magnesium
6. Total phenols
7. Flavanoids
8. Nonflavanoid phenols
9. Proanthocyanins
10. Color intensity
11. Hue
12. OD280/OD315 of diluted wines
13. Proline

The target variable is the class of wine (1, 2, or 3).

#### Requirements

1. Load the Wine dataset from scikit-learn
2. Preprocess the data (standardize features)
3. Split the data into training and testing sets
4. Build a multi-layer neural network using PyTorch with:
   - An input layer (matching the number of features)
   - At least one hidden layer with ReLU activation
   - An output layer with appropriate activation for classification
5. Train your model using an appropriate loss function and optimizer
6. Evaluate your model's performance on the test set
7. Experiment with different network architectures or hyperparameters to improve performance

## Submission

- Submit the URL of the GitHub Repository that contains your work to NTU black board.
- Should you reference the work of your classmate(s) or online resources, give them credit by adding either the name of your classmate or URL.


In [87]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load the Wine dataset
wine = load_wine()
X, y = wine.data, wine.target

# Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)


In [88]:
# Create TensorDatasets and DataLoaders
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [89]:
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

In [90]:
print(len(train_loader.dataset))

142


In [91]:
# Check the size of the data set
print(len(test_loader.dataset))

36


In [92]:

# Convert to a hashable form for row-wise comparison
train_set = set(map(tuple, X_train))
test_set = set(map(tuple, X_test))

# Check for data leakage
overlap = train_set & test_set
print(f"Number of overlapping samples: {len(overlap)}")

Number of overlapping samples: 0


In [93]:
# Check labels situation
for inputs, labels in train_loader:
    print(labels.unique())
    break

tensor([0, 1, 2])


In [94]:
# Define neural network
class WineNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(WineNet, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

In [95]:
import itertools

results = []
input_size = X_train.shape[1]
hidden_sizes = [8, 16]
learning_rates = [0.01, 0.001]
epochs = 100

for hidden_size, lr in itertools.product(hidden_sizes, learning_rates):
    model = WineNet(input_size, hidden_size, output_size=3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # --- training loop ---
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            labels = labels.long()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # --- evaluation ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            labels = labels.long()
            outputs = model(inputs)
            _, predicted = torch.max(outputs, dim=1)
            mismatches = (predicted != labels)
            if mismatches.any():
              print("Hyperparameters used:", "hidden_size", hidden_size, "lr", lr)
              print("Misclassified inputs:", inputs[mismatches])
              print("True labels:", labels[mismatches])
              print("Predicted:", predicted[mismatches])
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    results.append({'hidden_size': hidden_size, 'lr': lr, 'accuracy': accuracy})

for r in sorted(results, key=lambda x: -x['accuracy']):
    print(r)

Hyperparameters used: hidden_size 16 lr 0.001
Misclassified inputs: tensor([[-0.2849,  0.9817, -1.4129, -1.0495, -1.3861, -1.0657, -0.7824,  0.5491,
         -1.3332, -0.7172, -1.1295, -0.6945, -1.1938]])
True labels: tensor([1])
Predicted: tensor([2])
{'hidden_size': 8, 'lr': 0.01, 'accuracy': 100.0}
{'hidden_size': 8, 'lr': 0.001, 'accuracy': 100.0}
{'hidden_size': 16, 'lr': 0.01, 'accuracy': 100.0}
{'hidden_size': 16, 'lr': 0.001, 'accuracy': 97.22222222222223}


With only 36 test samples, it is possible to generalize 100% accuracy. Data leakage has been ruled out. There is no test data set overlapping with train data set. Only the smallest model + smallest learning rate combination underperformed. This makes sense: a smaller hidden layer (8 neurons vs 16) has less representational capacity, so it may need its weights to land in a more precise/narrow region to correctly separate the 3 classes. With hidden_size=16, there's more redundancy/capacity, so even a slower-moving lr=0.001 optimizer might find some good configuration within 100 epochs. With only 8 neurons, there's less margin for error, and the slower learning rate simply didn't get all the way there in time.